In [3]:
# 确保已安装必要的库: pip install rdkit-pypi
import re
from rdkit import Chem

def _get_char_level_hierarchy(smiles_string: str) -> str:
    """
    (Helper function) Generates a detailed, character-by-character structural label string
    for a given SMILES string. This is the basis for the component sequence.
    """
    if not smiles_string:
        return ""

    mol = Chem.MolFromSmiles(smiles_string)
    if not mol:
        return "!" * len(smiles_string)

    hierarchy = [''] * len(smiles_string)
    
    # Pass 1: Syntactically label branch characters '()' as 'B'
    branch_level = 0
    for i, char in enumerate(smiles_string):
        if char == '(':
            branch_level += 1
            hierarchy[i] = 'B'
        elif char == ')':
            hierarchy[i] = 'B'
            branch_level -= 1
        # Note: We no longer label atoms *inside* a branch as 'B' here.
        # They will be classified by their chemical nature in Pass 2.

    # Pass 2: Chemically label atoms and other characters
    aromatic_atom_indices = {atom.GetIdx() for atom in mol.GetAtoms() if atom.GetIsAromatic()}
    ring_info = mol.GetRingInfo()
    ring_atom_indices = set()
    for ring in ring_info.AtomRings():
        ring_atom_indices.update(ring)

    pattern = re.compile(r"(\[[^\]]+]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\\\|\/|:|~|@|\?|>|\*|\$|\%[0-9]{2}|[0-9])")
    tokens = pattern.findall(smiles_string)
    
    if "".join(tokens)!= smiles_string:
        # Fallback for rare tokenization failures
        return "".join([h if h else 'L' for h in hierarchy])

    atom_idx_counter = 0
    current_pos = 0
    non_atom_tokens = {'(', ')', '.', '=', '#', '-', '+', '\\', '/', ':', '~', '@', '?', '>', '*', '$'}

    for token in tokens:
        token_len = len(token)
        is_atom = not (token in non_atom_tokens or token.isdigit() or token.startswith('%'))

        if is_atom:
            is_aromatic = atom_idx_counter in aromatic_atom_indices
            is_in_ring = atom_idx_counter in ring_atom_indices
            for i in range(current_pos, current_pos + token_len):
                if hierarchy[i]: continue # Skip if already labeled (e.g., as 'B')
                if is_aromatic:
                    hierarchy[i] = 'A'  # Aromatic has top priority
                elif is_in_ring:
                    hierarchy[i] = 'R'
            atom_idx_counter += 1
        else:
            for i in range(current_pos, current_pos + token_len):
                if hierarchy[i]: continue
                if token.isdigit() or token.startswith('%'):
                    hierarchy[i] = 'R' # Ring closure digits
                # Other non-atom tokens (like bonds '=') are treated as part of linear chains
                else:
                    hierarchy[i] = 'L'
        
        current_pos += token_len

    final_hierarchy = [h if h else 'L' for h in hierarchy]
    return "".join(final_hierarchy)

def get_component_sequence(smiles_string: str) -> str:
    """
    Analyzes a SMILES string and returns a sequence of its high-level structural components.
    
    This function compresses the detailed character-level labels into a sequence of
    component tokens (e.g., 'A' for Aromatic, 'L' for Linear, 'B' for Branch).

    Args:
        smiles_string: The input SMILES string.

    Returns:
        A space-separated string of component tokens (e.g., "A L B L").
        Returns an empty string for invalid SMILES.
    """
    char_level_labels = _get_char_level_hierarchy(smiles_string)

    if not char_level_labels or "!" in char_level_labels:
        return ""  # Return empty for invalid SMILES

    if len(char_level_labels) == 0:
        return ""

    component_sequence = []
    # Use a simple regex to find consecutive groups of the same character
    # This elegantly handles the compression logic.
    for match in re.finditer(r'(.)\1*', char_level_labels):
        component_sequence.append(match.group(1)) # Append the character of the group
            
    return " ".join(component_sequence)

# --- 使用示例 ---
# Your example: c1ccccc1OCC(=O)N
# This SMILES has no parentheses, so it is parsed as an aromatic ring followed by a linear chain.
smiles1 = "c1ccccc1OCC(=O)N" 
# A more complex example with branches and rings
smiles2 = "CC(C)C1=CC=C(C=C1)C(C)C(=O)O" # Ibuprofen

hierarchy1 = get_component_sequence(smiles1)
hierarchy2 = get_component_sequence(smiles2)

print(f"SMILES:    {smiles1}")
print(f"Hierarchy: {hierarchy1}")
print("-" * 30)
print(f"SMILES:    {smiles2}")
print(f"Hierarchy: {hierarchy2}")

# --- 预期输出 ---
# SMILES:    c1ccccc1OCC(=O)N
# Hierarchy: A L
# ------------------------------
# SMILES:    CC(C)C1=CC=C(C=C1)C(C)C(=O)O
# Hierarchy: L B L A L B L

SMILES:    c1ccccc1OCC(=O)N
Hierarchy: A R A R L B L B L
------------------------------
SMILES:    CC(C)C1=CC=C(C=C1)C(C)C(=O)O
Hierarchy: L B L B A R L A L A B A L A R B L B L B L B L B L


In [4]:
import os, json, time
from tqdm import tqdm
from collections import defaultdict
from rdkit import Chem

PATT = {
    'HETEROATOM': '[!#6]',
    'DOUBLE_TRIPLE_BOND': '*=,#*',
    'ACETAL': '[CX4]([O,N,S])[O,N,S]'
}
PATT = {k: Chem.MolFromSmarts(v) for k, v in PATT.items()}


def get_fg_set(mol):
    """
    Identify FGs and convert to SMILES
    Args:
        mol:
    Returns: a set of FG's SMILES
    """
    fgs = []  # Function Groups

    # <editor-fold desc="identify and merge rings">
    rings = [set(x) for x in Chem.GetSymmSSSR(mol)]  # get simple rings
    flag = True  # flag == False: no rings can be merged
    while flag:
        flag = False
        for i in range(len(rings)):
            if len(rings[i]) == 0: continue
            for j in range(i + 1, len(rings)):
                shared_atoms = rings[i] & rings[j]
                if len(shared_atoms) > 2:
                    rings[i].update(rings[j])
                    rings[j] = set()
                    flag = True
    rings = [r for r in rings if len(r) > 0]
    # </editor-fold>

    # <editor-fold desc="identify functional atoms and merge connected ones">
    marks = set()
    for patt in PATT.values():  # mark functional atoms
        for sub in mol.GetSubstructMatches(patt):
            marks.update(sub)
    atom2fg = [[] for _ in range(mol.GetNumAtoms())]  # atom2fg[i]: list of i-th atom's FG idx
    for atom in marks:  # init: each marked atom is a FG
        fgs.append({atom})
        atom2fg[atom] = [len(fgs)-1]
    for bond in mol.GetBonds():  # merge FGs
        if bond.IsInRing(): continue
        a1, a2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        if a1 in marks and a2 in marks:  # a marked atom should only belong to a FG, if atoms are both marked, merge their FGs into a FG
            assert a1 != a2
            assert len(atom2fg[a1]) == 1 and len(atom2fg[a2]) == 1
            # merge a2' FG to a1's FG
            fgs[atom2fg[a1][0]].update(fgs[atom2fg[a2][0]])
            fgs[atom2fg[a2][0]] = set()
            atom2fg[a2] = atom2fg[a1]
        elif a1 in marks:  # only one atom is marked, add neighbour atom to its FG as its environment
            assert len(atom2fg[a1]) == 1
            # add a2 to a1's FG
            fgs[atom2fg[a1][0]].add(a2)
            atom2fg[a2].extend(atom2fg[a1])
        elif a2 in marks:
            # add a1 to a2's FG
            assert len(atom2fg[a2]) == 1
            fgs[atom2fg[a2][0]].add(a1)
            atom2fg[a1].extend(atom2fg[a2])
        else:  # both atoms are unmarked, i.e. a trivial C-C single bond
            # add single bond to fgs
            fgs.append({a1, a2})
            atom2fg[a1].append(len(fgs)-1)
            atom2fg[a2].append(len(fgs)-1)

    tmp = []
    for fg in fgs:
        if len(fg) == 0: continue
        if len(fg) == 1 and mol.GetAtomWithIdx(list(fg)[0]).IsInRing(): continue
        tmp.append(fg)
    fgs = tmp
    # </editor-fold>

    fgs.extend(rings)  # final FGs: rings + FGs (not in rings)

    fg_smiles = set()
    for fg in fgs:
        fg_smiles.add(Chem.MolFragmentToSmiles(mol, fg))

    return fg_smiles

In [9]:
import numpy as np
from scipy.sparse import csr_matrix
import networkx as nx
from rdkit import Chem
from collections import defaultdict


# feature dim
ATOM_DIM = 101
BOND_DIM = 11
FG_DIM = 73
FG_EDGE_DIM = ATOM_DIM

ALLOWABLE_BOND_FEATURES = {
    'bond_type': ['SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC'],
    'conjugated': ['T/F'],
    'stereo': ['STEREONONE', 'STEREOZ', 'STEREOE', 'STEREOCIS', 'STEREOTRANS', 'STEREOANY']
}

PATT = {
    'HETEROATOM': '[!#6]',
    'DOUBLE_TRIPLE_BOND': '*=,#*',
    'ACETAL': '[CX4]([O,N,S])[O,N,S]'
}
PATT = {k: Chem.MolFromSmarts(v) for k, v in PATT.items()}


def one_of_k_encoding(x, allowable_set):
    if x not in allowable_set:
        raise Exception("input {0} not in allowable set{1}:".format(x, allowable_set))
    return list(map(lambda s: x == s, allowable_set))


def one_of_k_encoding_unk(x, allowable_set):
    """Maps inputs not in the allowable set to the last element."""
    if x not in allowable_set:
        x = allowable_set[-1]
    return list(map(lambda s: x == s, allowable_set))


def get_atom_feature(atom):
    return np.array(
        one_of_k_encoding_unk(atom.GetSymbol(), [
            'C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca', 'Fe', 'As', 'Al', 'I', 'B',
            'V', 'K', 'Tl', 'Yb', 'Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti', 'Zn', 'H', 'Li', 'Ge', 'Cu',
            'Au', 'Ni', 'Cd', 'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb', 'Unknown'
        ]) +
        one_of_k_encoding(atom.GetDegree(), [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]) +
        one_of_k_encoding_unk(atom.GetTotalNumHs(), [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]) +
        one_of_k_encoding_unk(atom.GetImplicitValence(), [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]) +
        one_of_k_encoding_unk(atom.GetTotalValence(), [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]) +
        one_of_k_encoding_unk(atom.GetFormalCharge(), [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]) +
        [atom.GetIsAromatic()] +
        [atom.IsInRing()]
    )


def get_bond_feature(bond):
    return np.array(
        one_of_k_encoding(str(bond.GetBondType()), ALLOWABLE_BOND_FEATURES['bond_type']) +
        [bond.GetIsConjugated()] +
        one_of_k_encoding(str(bond.GetStereo()), ALLOWABLE_BOND_FEATURES['stereo'])
    )


def get_fg_feature(fg_prop):
    return np.array(
        one_of_k_encoding_unk(fg_prop['#C'], range(11)) +  # 0-10, 10+
        one_of_k_encoding_unk(fg_prop['#O'], range(6)) +  # 0-5, 5+
        one_of_k_encoding_unk(fg_prop['#N'], range(6)) +
        one_of_k_encoding_unk(fg_prop['#P'], range(6)) +
        one_of_k_encoding_unk(fg_prop['#S'], range(6)) +
        [fg_prop['#X'] > 0] +
        [fg_prop['#UNK'] > 0] +
        one_of_k_encoding_unk(fg_prop['#SINGLE'], range(11)) +  # 0-10, 10+
        one_of_k_encoding_unk(fg_prop['#DOUBLE'], range(8)) +  # 0-6, 6+
        one_of_k_encoding_unk(fg_prop['#TRIPLE'], range(8)) +
        one_of_k_encoding_unk(fg_prop['#AROMATIC'], range(8)) +
        [fg_prop['IsRing']]
    )


def mol_to_graphs(mol):
    fgs = []  # Function Groups

    # <editor-fold desc="identify and merge rings">
    rings = [set(x) for x in Chem.GetSymmSSSR(mol)]  # get simple rings
    flag = True  # flag == False: no rings can be merged
    while flag:
        flag = False
        for i in range(len(rings)):
            if len(rings[i]) == 0: continue
            for j in range(i+1, len(rings)):
                shared_atoms = rings[i] & rings[j]
                if len(shared_atoms) > 2:
                    rings[i].update(rings[j])
                    rings[j] = set()
                    flag = True
    rings = [r for r in rings if len(r) > 0]
    # </editor-fold>

    # <editor-fold desc="identify functional atoms and merge connected ones">
    marks = set()
    for patt in PATT.values():  # mark functional atoms
        for sub in mol.GetSubstructMatches(patt):
            marks.update(sub)
    atom2fg = [[] for _ in range(mol.GetNumAtoms())]  # atom2fg[i]: list of i-th atom's FG idx
    for atom in marks:  # init: each marked atom is a FG
        fgs.append({atom})
        atom2fg[atom] = [len(fgs)-1]
    for bond in mol.GetBonds():  # merge FGs
        if bond.IsInRing(): continue
        a1, a2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        if a1 in marks and a2 in marks:  # a marked atom should only belong to a FG, if atoms are both marked, merge their FGs into a FG
            assert a1 != a2
            assert len(atom2fg[a1]) == 1 and len(atom2fg[a2]) == 1
            # merge a2' FG to a1's FG
            fgs[atom2fg[a1][0]].update(fgs[atom2fg[a2][0]])
            fgs[atom2fg[a2][0]] = set()
            atom2fg[a2] = atom2fg[a1]
        elif a1 in marks:  # only one atom is marked, add neighbour atom to its FG as its environment
            assert len(atom2fg[a1]) == 1
            # add a2 to a1's FG
            fgs[atom2fg[a1][0]].add(a2)
            atom2fg[a2].extend(atom2fg[a1])
        elif a2 in marks:
            # add a1 to a2's FG
            assert len(atom2fg[a2]) == 1
            fgs[atom2fg[a2][0]].add(a1)
            atom2fg[a1].extend(atom2fg[a2])
        else:  # both atoms are unmarked, i.e. a trivial C-C single bond
            # add single bond to fgs
            fgs.append({a1, a2})
            atom2fg[a1].append(len(fgs)-1)
            atom2fg[a2].append(len(fgs)-1)
    tmp = []
    for fg in fgs:
        if len(fg) == 0: continue
        if len(fg) == 1 and mol.GetAtomWithIdx(list(fg)[0]).IsInRing(): continue  # single atom FGs: 1. marked atom only in ring: remove; 2. ion or simple substance: retain
        tmp.append(fg)
    fgs = tmp
    # </editor-fold>

    fgs.extend(rings)  # final FGs: rings + FGs (not in rings)
    atom2fg = [[] for _ in range(mol.GetNumAtoms())]
    for i in range(len(fgs)): # update atom2fg
        for atom in fgs[i]:
            atom2fg[atom].append(i)

    # <editor-fold desc="generate atom-level graph and get FG's properties">
    atom_features, bond_list, bond_features = [], [], []
    fg_prop = [defaultdict(int) for _ in range(len(fgs))]  # prop: atom: #C, #O, #N, #P, #S, #X, #UNK; bond: #SINGLE, #DOUBLE, #TRIPLE, #AROMATIC, IsRing
    for atom in mol.GetAtoms():
        atom_features.append(get_atom_feature(atom).tolist())
        elem = atom.GetSymbol()
        if elem in ['C', 'O', 'N', 'P', 'S']:
            key = '#'+elem
        elif elem in ['F', 'Cl', 'Br', 'I']:
            key = '#X'
        else:
            key = '#UNK'
        for fg_idx in atom2fg[atom.GetIdx()]:
            fg_prop[fg_idx][key] += 1
    for bond in mol.GetBonds():
        a1, a2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bond_list.extend([[a1, a2], [a2, a1]])
        bond_features.extend([get_bond_feature(bond).tolist()] * 2)
        key = '#'+str(bond.GetBondType())
        for fg_idx in (set(atom2fg[a1]) & set(atom2fg[a2])):
            fg_prop[fg_idx][key] += 1
            if bond.IsInRing():
                fg_prop[fg_idx]['IsRing'] = 1
    # </editor-fold>

    # <editor-fold desc="generate FG-level graph">
    fg_features, fg_edge_list, fg_edge_features = [], [], []
    for i in range(len(fgs)):
        fg_features.append(get_fg_feature(fg_prop[i]).tolist())
        for j in range(i+1, len(fgs)):
            shared_atoms = list(fgs[i] & fgs[j])
            if len(shared_atoms) > 0:
                fg_edge_list.extend([[i, j], [j, i]])
                if len(shared_atoms) == 1:
                    fg_edge_features.extend([atom_features[shared_atoms[0]]] * 2)
                else:  # two rings shared 2 atoms, i.e. 1 edge
                    assert len(shared_atoms) == 2
                    ef = [(i+j)/2 for i, j in zip(atom_features[shared_atoms[0]], atom_features[shared_atoms[1]])]
                    fg_edge_features.extend([ef] * 2)
    # </editor-fold>

    atom2fg_list = []
    for fg_idx in range(len(fgs)):
        for atom_idx in fgs[fg_idx]:
            atom2fg_list.append([atom_idx, fg_idx])

    return atom_features, bond_list, bond_features, fg_features, fg_edge_list, fg_edge_features, atom2fg_list, fg_prop


if __name__ == '__main__':
    smiles = 'C[C@@H](O[C@H]1OCCN(CC2=NNC(=O)N2)[C@H]1C1=CC=C(F)C=C1)C1=CC(=CC(=C1)C(F)(F)F)C(F)(F)F'
    mol = Chem.MolFromSmiles(smiles)
    atom_features, bond_list, bond_features, fg_features, fg_edge_list, fg_edge_features, atom2fg_list, fg_prop = mol_to_graphs(mol)
    pass

[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValence(getExplicit=False)
[05:15:05] DEPRECATION WARNING: please use GetValen

In [10]:
fg_prop

[defaultdict(int,
             {'#C': 1,
              '#X': 1,
              '#SINGLE': 1,
              '#O': 0,
              '#N': 0,
              '#P': 0,
              '#S': 0,
              '#UNK': 0,
              '#DOUBLE': 0,
              '#TRIPLE': 0,
              '#AROMATIC': 0,
              'IsRing': 0}),
 defaultdict(int,
             {'#C': 2,
              '#O': 1,
              '#SINGLE': 2,
              '#N': 0,
              '#P': 0,
              '#S': 0,
              '#X': 0,
              '#UNK': 0,
              '#DOUBLE': 0,
              '#TRIPLE': 0,
              '#AROMATIC': 0,
              'IsRing': 0}),
 defaultdict(int,
             {'#C': 1,
              '#X': 1,
              '#SINGLE': 1,
              '#O': 0,
              '#N': 0,
              '#P': 0,
              '#S': 0,
              '#UNK': 0,
              '#DOUBLE': 0,
              '#TRIPLE': 0,
              '#AROMATIC': 0,
              'IsRing': 0}),
 defaultdict(int,
          

In [5]:
from rdkit import Chem
from rdkit.Chem import AllChem

mol = Chem.MolFromSmiles("c1ccccc1OCC(=O)N")

In [6]:
get_fg_set(mol)

{'CC(N)=O', 'c1ccccc1', 'cOC'}